In [67]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims
import numpy as np
import pandas as pd
import os 

In [86]:
dataset =  'ACCpatterns'
region = 'CINGULATE' #"S.C.-sylv." "S.T.s." "CINGULATE"
side = 'R' #"L"

In [87]:
labels_ACCP = pd.read_csv("/neurospin/dico/data/bv_databases/human/partially_labeled/ACCpatterns/all.csv")
sub_part = pd.read_csv("/neurospin/dico/data/deep_folding/current/datasets/ACCpatterns/ACCpatterns_all_subs.csv", header=None) 
labels_ACCP = labels_ACCP[labels_ACCP['long_name'].isin(sub_part[0].to_list())]
restriction = labels_ACCP[(labels_ACCP['DATABASE'] == "NIMH_COS") | (labels_ACCP['DATABASE'] =="NIMH_COSSIB") | (labels_ACCP['DATABASE'] =="NIMH_NV")]
restriction = restriction[['long_name', 'PCS_R1', 'PCS_L1', 'PCS_R_num3_1', 'PCS_L_num3_1']].dropna()
restriction

,long_name,PCS_R1,PCS_L1,PCS_R_num3_1,PCS_L_num3_1
0,nih_chp_04701_t1,abs,pre,0.0,1.0
1,nih_chp_01534_t1,pre,pro,1.0,2.0
2,nih_chp_04623_t1,pro,pre,2.0,1.0
3,nih_chp_01503_t1,abs,abs,0.0,0.0
4,nih_chp_00404_t1,pre,pro,1.0,2.0
...,...,...,...,...,...
117,nih_chp_05116_t1,abs,abs,0.0,0.0
118,nih_chp_05147_t1,pre,abs,1.0,0.0
119,nih_chp_05032_t1,abs,abs,0.0,0.0
120,nih_chp_05190_t1,abs,abs,0.0,0.0


In [88]:
sample = restriction[restriction['PCS_R_num3_1'] == 0].long_name.to_list()
len(sample)
sample = sample[21:42]

In [89]:
volume=True
nb_columns = 7 
block = a.createWindowsBlock(nb_columns) # nb of columns
dic_windows = {}

bucket_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}buckets'
bucket_files = []

for subject_id in sample:
    bck_path = f'{bucket_path}/{subject_id}_cropped_skeleton.bck'
    if os.path. isfile(bck_path):
        bucket_files.append(bck_path)
    else:
        print(f"{bck_path} is not a correct path, or the .bck doesn't exist")

for i, file in enumerate(bucket_files):
    dic_windows[f'bck_{i}'] = a.loadObject(file)
    dic_windows[f'w_{i}'] = a.createWindow('3D', block=block)#geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
    dic_windows[f'w_{i}'].addObjects(dic_windows[f'bck_{i}'])

/neurospin/dico/data/deep_folding/current/datasets/ACCpatterns/crops/2mm/CINGULATE/mask/Rbuckets/nih_chp_02955_t1_cropped_skeleton.bck is not a correct path, or the .bck doesn't exist
/neurospin/dico/data/deep_folding/current/datasets/ACCpatterns/crops/2mm/CINGULATE/mask/Rbuckets/nih_chp_04830_t1_cropped_skeleton.bck is not a correct path, or the .bck doesn't exist
/neurospin/dico/data/deep_folding/current/datasets/ACCpatterns/crops/2mm/CINGULATE/mask/Rbuckets/nih_chp_04778_t1_cropped_skeleton.bck is not a correct path, or the .bck doesn't exist
/neurospin/dico/data/deep_folding/current/datasets/ACCpatterns/crops/2mm/CINGULATE/mask/Rbuckets/nih_chp_03858_t1_cropped_skeleton.bck is not a correct path, or the .bck doesn't exist
/neurospin/dico/data/deep_folding/current/datasets/ACCpatterns/crops/2mm/CINGULATE/mask/Rbuckets/nih_chp_05776_t1_cropped_skeleton.bck is not a correct path, or the .bck doesn't exist
/neurospin/dico/data/deep_folding/current/datasets/ACCpatterns/crops/2mm/CINGULA

In [90]:
volume=True
nb_columns=7
block = a.createWindowsBlock(nb_columns) # nb of columns
dic_windows = {}

referential1 = a.createReferential()

mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'
# dic_windows['Sulci_color']=a.loadObject('/casa/host/build/share/brainvisa-share-5.2/nomenclature/hierarchy/sulcal_root_colors.hie')
dic_windows['Sulci_color'] = os.path.join(aims.carto.Paths.shfjShared(), 'nomenclature',
                                          'hierarchy', 'sulcal_root_colors.hie')

for i, subject_id in enumerate(sample):
    volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
    
    if os.path.isfile(volume_path):
        vol = aims.read(volume_path)
        
        dic_windows[f'a_vol{nb_columns*i}'] = a.toAObject(vol)
        #dic_windows[f'a_vol{i}'].setPalette(absoluteMode=True)
        dic_windows[f'rvol{nb_columns*i}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{nb_columns*i}']], method='VolumeRenderingFusionMethod')
        dic_windows[f'rvol{nb_columns*i}'].releaseAppRef()
        dic_windows[f'rvol{nb_columns*i}'].assignReferential(referential1)
        dic_windows[f'wvr{nb_columns*i}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
        dic_windows[f'wvr{nb_columns*i}'].addObjects(dic_windows[f'rvol{nb_columns*i}'])
    else:
        print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")

    # if dataset == 'hcp':
    #     path_to_t1mri = f'/neurospin/dico/data/bv_databases/human/not_labeled/hcp/hcp/{subject_id}/t1mri/BL'
    #     white_matter_path = f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject_id}_{side}white.gii'
    #     sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/{side}{subject_id}.arg'
    #     spam_labelled_sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/spam_auto/{side}{subject_id}_spam_auto.arg'
    #     deep_labelled_sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/deepcnn_auto/{side}{subject_id}_deepcnn_auto.arg' 

    # elif dataset == 'ACCpatterns':
    #     path_to_t1mri = f'/neurospin/dico/data/bv_databases/human/partially_labeled/ACCpatterns/all/{subject_id}/t1mri/default_acquisition'
    #     white_matter_path = f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject_id}_{side}white.gii'
    #     sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/default_session_auto/{side}{subject_id}_default_session_auto.arg'
    #     spam_labelled_sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/spam_auto/{side}{subject_id}_spam_auto.arg'
    #     deep_labelled_sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/deepcnn_auto/{side}{subject_id}_deepcnn_auto.arg' 

    # if os.path.isfile(white_matter_path):
    #     # To visualize the white matter for specific people
    #     dic_windows[f'white_{subject_id}'] = a.loadObject(white_matter_path)
    #     #dic_windows[f'white_{subject_id}'].loadReferentialFromHeader()
    #     dic_windows[f'white_{subject_id}'].assignReferential(referential1)
    # else:
    #     print(f"{white_matter_path} is not a correct path, or the .white.gii doesn't exist")
    
    # if os.path.isfile(spam_labelled_sulci_path):
    #     # To visualize the annotated sulci for specific people
    #     dic_windows[f'sulci_{subject_id}'] = a.loadObject(spam_labelled_sulci_path)
    #     #dic_windows[f'sulci_{subject_id}'].loadReferentialFromHeader()
    #     dic_windows[f'sulci_{subject_id}'].assignReferential(referential1)
    # else:
    #     print(f"{spam_labelled_sulci_path} is not a correct path, or the .arg doesn't exist")
    #     print("Automatic try with 'deepcnn_session_auto' instead of 'spam_session_auto'")
    #     if  os.path.isfile(deep_labelled_sulci_path):
    #         # To visualize the annotated sulci for specific people
    #         dic_windows[f'sulci_{subject_id}'] = a.loadObject(deep_labelled_sulci_path)
    #         #dic_windows[f'sulci_{subject_id}'].loadReferentialFromHeader()
    #         dic_windows[f'sulci_{subject_id}'].assignReferential(referential1)
    #     else:
    #         print(f"{deep_labelled_sulci_path} is not a correct path, or the .arg doesn't exist")
    #         print("Try with non labeled sulci")
    #         if os.path.isfile(sulci_path):
    #             # To visualize the sulci for specific people
    #             dic_windows[f'sulci_{subject_id}'] = a.loadObject(sulci_path)
    #             dic_windows[f'sulci_{subject_id}'].loadReferentialFromHeader()
    #         else:
    #             print(f"{sulci_path} is not a correct path, or the .arg doesn't exist")
    
    #dic_windows[f'wvr{nb_columns*i+1}'] = a.createWindow('3D', block=block)
    #dic_windows[f'wvr{nb_columns*i+1}'].addObjects(dic_windows[f'white_{subject_id}'])
    #dic_windows[f'wvr{nb_columns*i+1}'].addObjects(dic_windows[f'sulci_{subject_id}'])

/neurospin/dico/data/bv_databases/human/partially_labeled/ACCpatterns/all/nih_chp_02955_t1/t1mri/default_acquisition/default_analysis/segmentation/mesh/nih_chp_02955_t1_Rwhite.gii is not a correct path, or the .white.gii doesn't exist
/neurospin/dico/data/bv_databases/human/partially_labeled/ACCpatterns/all/nih_chp_02955_t1/t1mri/default_acquisition/default_analysis/folds/3.1/spam_auto/Rnih_chp_02955_t1_spam_auto.arg is not a correct path, or the .arg doesn't exist
Automatic try with 'deepcnn_session_auto' instead of 'spam_session_auto'
/neurospin/dico/data/bv_databases/human/partially_labeled/ACCpatterns/all/nih_chp_02955_t1/t1mri/default_acquisition/default_analysis/folds/3.1/deepcnn_auto/Rnih_chp_02955_t1_deepcnn_auto.arg is not a correct path, or the .arg doesn't exist
Try with non labeled sulci
memory limit: 44059243315
Reading FGraph version 3.1
bounding box found : 57, 46, 18
                     138, 225, 93
nifti transfo: 2
/neurospin/dico/data/bv_databases/human/partially_lab